# Intyoza: offline chat and review

**Notebook:** 2 of 4. Runs the trained chatbot on the laptop without internet, and reviews what it could not answer.

## How a message is handled

Intyoza is a learning platform first: a young person asks *"Agakingirizo ni iki?"* or *"Imihango ni iki?"* and gets a clear answer in Kinyarwanda, then keeps learning.

1. **Normalise** the text (same rules as in training).
2. **Learning questions are answered first.** A clear match to a learning question always gets its approved answer, adapted to the age group, with where to go for services and the next questions in the learning path. If the message also looks risky, help contacts are added under the answer (or above it when the danger is very clear).
3. **Safety net:** a message about danger (violence, self-harm, medical emergency) that is not a learning question gets help contacts (112, 114, 116, Isange 3029) and supportive follow-up.
4. **Greetings and thanks** get a friendly reply and the learning menu.
5. **"Did you mean":** a possible match gets the three closest questions to choose from.
6. **Documents:** if no approved answer matches, the trusted documents are searched (when the document index is present).
7. Otherwise the topic menu is offered, and the message is saved to `intyoza_unanswered.csv` for the team to review.

## Contents

1. Setup
2. Load the chatbot
3. Model card
4. Scripted test
5. Interactive chat
6. Review unanswered questions

**Requirement:** the `models` folder from `1_Train.ipynb` (or `intyoza_models.zip` unzipped here).

## 1. Setup

| Setting | Meaning |
|---|---|
| `AGE_GROUP` | `10-14`, `15-19` or `20-24`; some answers have an age-specific version |
| `USE_DOCUMENTS` | Use the document index as a fallback (slower start); `False` for approved answers only |
| `REPLY_LANGUAGE` | `rw` always answers in Kinyarwanda; `auto` answers English messages in English |

In [ ]:
import os, sys, json
import pandas as pd
FOLDER = os.getcwd()
assert os.path.exists(os.path.join(FOLDER, "intyoza_core.py")), "Open this notebook from inside the Intyoza folder."
sys.path.insert(0, FOLDER)
MODELS = os.path.join(FOLDER, "models")
UNANSWERED = os.path.join(FOLDER, "intyoza_unanswered.csv")
pd.set_option("display.max_colwidth", 120)

AGE_GROUP = "15-19"
USE_DOCUMENTS = True
REPLY_LANGUAGE = "rw"

## 2. Load the chatbot

Loads the fine-tuned matcher, the small models and the question index. All Hugging Face libraries are forced offline. The first start after training builds the question index cache (a few minutes on a laptop); after that it loads in seconds.

In [ ]:
from intyoza_core import load_bot, run_chat
bot = load_bot(MODELS, FOLDER, age_group=AGE_GROUP, reply_lang=REPLY_LANGUAGE, use_documents=USE_DOCUMENTS,
               log_path=UNANSWERED)
print(f"Intyoza loaded offline: {len(bot.qa)} approved questions | answer cut-offs LO={bot.lo}, HI={bot.hi} | "
      f"documents: {'on' if bot.docs else 'off'}")

## 3. Model card

The results recorded when these models were trained (written by step 17 of `1_Train.ipynb`).

In [ ]:
card_path = os.path.join(MODELS, "model_card.json")
if os.path.exists(card_path):
    card = json.load(open(card_path, encoding="utf-8"))
    m = card["metrics"]
    rows = {"trained at": card["trained_at"], "questions": card["data"]["questions"],
            "phrasings": card["data"]["phrasings"],
            "top-1 accuracy (fine-tuned)": f"{m['fine_tuned']['top1']:.1%}",
            "top-3 accuracy (fine-tuned)": f"{m['fine_tuned']['top3']:.1%}",
            "risky messages caught": f"{m['safety']['recall']:.1%}",
            "false alarm rate": f"{m['safety']['false_alarm_rate']:.1%}"}
    display(pd.Series(rows, name="value").to_frame())
else:
    print("No model card found (models trained with an older version of 1_Train.ipynb).")

## 4. Scripted test

A fixed set of messages, run on every new model before it is used, so changes in behaviour are easy to spot.

In [ ]:
SCRIPT = [
    "mwiriwe",
    "agakingirizo ni iki?",
    "imihango ni iki?",
    "virusi itera sida ni iki?",
    "ubwangavu ni iki?",
    "ni ryari umukobwa atangira imihango?",
    "agakingirizo gakoreshwa gute?",
    "ni he nabona serivisi zo kuboneza urubyaro ku buntu?",
    "umukunzi wanjye arampatira imibonano",
    "ndumva ntameze neza",
    "murakoze",
]
rows = []
for msg in SCRIPT:
    r = bot.reply(msg)
    rows.append({"message": msg, "type": r["type"], "help": r.get("help", ""), "reply": r["text"][:180].replace("\n", " "),
                 "options": " | ".join(r.get("options", [])[:3])})
pd.DataFrame(rows)

## 5. Interactive chat

Type a message and press Enter. Commands: a number picks an option, `quiz` starts the myth-or-fact game, `age 10-14` / `age 15-19` / `age 20-24` changes the age group, `exit` stops. Set `DEBUG = True` to see language, intent, risk level and match score for each message.

In [ ]:
DEBUG = False
bot.reset()            # new conversation (the scripted test above is forgotten)
run_chat(bot, unanswered_csv=UNANSWERED, debug=DEBUG)

## 6. Review unanswered questions

Every message the bot could not answer from the approved list is saved with its closest question and score. Reviewing them regularly is how the dataset grows:

- a message that should have matched an existing question: add it as a phrasing in `knowledge/Intyoza_Dataset.xlsx`
- a new common question: add it as a new question with an approved answer
- then run step 19 of `1_Train.ipynb` (refresh) or retrain

In [ ]:
if os.path.exists(UNANSWERED):
    u = pd.read_csv(UNANSWERED)
    print(f"{len(u)} unanswered messages saved; {int(u['answered_from_documents'].sum())} were answered from documents.")
    u["closest question"] = u["best_match"].map(lambda i: bot.qa[i]["q_rw"] or bot.qa[i]["q_en"] if i in bot.qa else "")
    display(u.tail(30))
else:
    print("No unanswered messages yet.")